# Step 1: Mapping Degraded Land in Indonesia
## For Solar PV Potential Assessment

**Goal:** Identify spatially degraded/marginal land in Indonesia that is suitable for utility-scale Solar PV — avoiding productive land, forests, and protected areas.

### Degraded Land Criteria (this notebook)
| Layer | Source | Logic |
|---|---|---|
| Bare / sparse vegetation | ESA WorldCover 2021 | Class 60 |
| Shrubland (degraded forest) | ESA WorldCover 2021 | Class 20 |
| Grassland / Imperata | ESA WorldCover 2021 | Class 30 |
| Deforested, no recovery | Hansen GFC 2023 | Forest loss (2001–2023) AND current cover ≠ forest |

### Exclusion Zones
- Protected areas (WDPA)
- Slope > 5° (SRTM)
- Cropland, built-up, water, wetland, mangrove

### Data Sources
- [ESA WorldCover 2021 v200](https://esa-worldcover.org) — 10 m land cover
- [Hansen Global Forest Change v1.11](https://glad.earthengine.app/view/global-forest-change) — 30 m deforestation
- [WDPA](https://www.protectedplanet.net) — protected areas
- [SRTM](https://www2.jpl.nasa.gov/srtm/) — elevation / slope

In [11]:
# ── 0. Install dependencies (run once) ──────────────────────────────────────
#!pip install geemap earthengine-api -q

In [12]:
# ── 1. Imports & Google Earth Engine Initialization ─────────────────────────
import os
import ee
import geemap
import pandas as pd

# Create a custom credentials directory
creds_dir = os.path.expanduser('~/earthengine_creds')
os.makedirs(creds_dir, exist_ok=True)
os.environ['EARTHENGINE_CACHE'] = creds_dir

# First time: run ee.Authenticate() and follow the browser prompt.
# After that, only ee.Initialize() is needed.
try:
    ee.Initialize(project='rachman-research')   # ← replace with your GCP project
    print("GEE initialized.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project='rachman-research')
    print("GEE authenticated and initialized.")

GEE initialized.


In [13]:
# ── 2. Area of Interest: Indonesia ──────────────────────────────────────────
indonesia = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq('country_na', 'Indonesia'))
)
aoi = indonesia.geometry()

# Quick check
print("Indonesia feature count:", indonesia.size().getInfo())
print("Bounding box:", aoi.bounds().getInfo()['coordinates'])

Indonesia feature count: 1
Bounding box: [[[95.00708009748928, -10.931678540191342], [141.02207260949848, -10.931678540191342], [141.02207260949848, 5.916250264699178], [95.00708009748928, 5.916250264699178], [95.00708009748928, -10.931678540191342]]]


In [14]:
# ── 3. ESA WorldCover 2021 (10 m) ───────────────────────────────────────────
# ESA WorldCover class legend:
#   10  Tree cover          20  Shrubland         30  Grassland
#   40  Cropland            50  Built-up           60  Bare/sparse veg
#   70  Snow/ice            80  Permanent water    90  Herbaceous wetland
#   95  Mangroves          100  Moss/lichen

worldcover = ee.ImageCollection("ESA/WorldCover/v200").first().clip(aoi)

# ── Degraded land layers from WorldCover ────────────────────────────────────
shrubland  = worldcover.eq(20)          # degraded/secondary forest indicator
grassland  = worldcover.eq(30)          # often Imperata cylindrica in Indonesia
bare_land  = worldcover.eq(60)          # highly degraded, mining spoils, etc.

degraded_wc = shrubland.Or(grassland).Or(bare_land).rename('degraded_wc')

# ── Exclusion masks from WorldCover ─────────────────────────────────────────
is_cropland = worldcover.eq(40)
is_buildup  = worldcover.eq(50)
is_water    = worldcover.eq(80)
is_wetland  = worldcover.eq(90)
is_mangrove = worldcover.eq(95)
is_forest   = worldcover.eq(10)

print("WorldCover loaded. Degraded land layer defined (shrubland + grassland + bare).")

WorldCover loaded. Degraded land layer defined (shrubland + grassland + bare).


In [15]:
# ── 4. Hansen Global Forest Change 2023 v1.11 (30 m) ───────────────────────
# Captures deforestation 2001–2023 at 30 m resolution.
# We want pixels that: (a) had ≥ 30% tree cover in 2000, (b) experienced
# forest loss in any year 2001–2023, and (c) are NOT currently forest cover.
# These are "deforested without recovery" — a key degraded land class.

hansen = ee.Image("UMD/hansen/global_forest_change_2023_v1_11").clip(aoi)

had_forest_2000    = hansen.select('treecover2000').gte(30)  # ≥ 30% canopy
lost_forest        = hansen.select('lossyear').gt(0)         # any loss event
currently_no_forest = is_forest.Not()                        # not forest today

deforested_no_recovery = (
    had_forest_2000
    .And(lost_forest)
    .And(currently_no_forest)
    .rename('deforested_no_recovery')
)

print("Hansen deforestation layer defined.")

Hansen deforestation layer defined.


In [16]:
# ── 5. Combine All Degraded Land Sources ────────────────────────────────────
degraded_all = (
    degraded_wc                  # shrubland + grassland + bare (ESA)
    .Or(deforested_no_recovery)  # deforested without recovery (Hansen)
    .rename('degraded_all')
)

print("Combined degraded land layer ready.")

Combined degraded land layer ready.


In [17]:
# ── 6. Exclusion Zones ───────────────────────────────────────────────────────

# 6a. Slope > 5° from SRTM DEM (solar farms need relatively flat terrain)
srtm  = ee.Image("USGS/SRTMGL1_003").clip(aoi)
slope = ee.Terrain.slope(srtm)
steep = slope.gt(5)   # exclude pixels steeper than 5 degrees

# 6b. Protected areas from WDPA (World Database of Protected Areas)
#     Rasterized at 100 m — pixels touching a protected area are excluded.
wdpa = (
    ee.FeatureCollection("WCMC/WDPA/current/polygons")
    .filter(ee.Filter.eq('PARENT_ISO3', 'IDN'))
)
protected = ee.Image(0).paint(wdpa, 1).unmask(0).rename('protected')

# 6c. Combined exclusion mask (any of these → exclude)
exclusion = (
    steep
    .Or(protected)
    .Or(is_cropland)
    .Or(is_buildup)
    .Or(is_water)
    .Or(is_wetland)
    .Or(is_mangrove)
)

print("Exclusion zones built: slope >5°, WDPA, cropland, built-up, water, wetland, mangrove.")

Exclusion zones built: slope >5°, WDPA, cropland, built-up, water, wetland, mangrove.


In [18]:
# ── 7. Final Degraded Land Layer (Solar-Suitable) ───────────────────────────
# = degraded land  AND  NOT in any exclusion zone
final_degraded = (
    degraded_all
    .And(exclusion.Not())
    .selfMask()                   # mask zeros so map is clean
    .rename('degraded_suitable')
)

# Label each pixel by its degradation source (useful for breakdown maps)
degradation_type = (
    ee.Image(0)
    .where(shrubland.And(exclusion.Not()),          1)   # shrubland
    .where(grassland.And(exclusion.Not()),          2)   # grassland
    .where(bare_land.And(exclusion.Not()),          3)   # bare land
    .where(deforested_no_recovery.And(exclusion.Not()), 4)  # deforested
    .selfMask()
    .rename('degradation_type')
)
# type codes: 1=shrubland, 2=grassland, 3=bare, 4=deforested-no-recovery

print("Final degraded land layer ready.")

Final degraded land layer ready.


In [19]:
# ── 8. Interactive Map ───────────────────────────────────────────────────────
Map = geemap.Map(center=[-2.5, 118], zoom=5)
Map.setOptions('HYBRID')

# WorldCover base (optional reference layer)
wc_vis = {
    'bands': ['Map'], 'min': 10, 'max': 100,
    'palette': ['006400','ffbb22','ffff4c','f096ff',
                'fa0000','b4b4b4','f0f0f0','0064c8',
                '0096a0','00cf75','fae6a0']
}
Map.addLayer(worldcover, wc_vis, 'ESA WorldCover 2021', False)

# Hansen forest loss
Map.addLayer(
    hansen.select('lossyear').selfMask(),
    {'min': 1, 'max': 23, 'palette': ['yellow', 'red']},
    'Hansen Forest Loss 2001–2023', False
)

# Exclusion zones
Map.addLayer(
    exclusion.selfMask(),
    {'palette': ['#808080'], 'min': 1, 'max': 1},
    'Exclusion Zones', False
)

# Degradation type map (all degraded, before exclusion)
type_vis = {
    'min': 1, 'max': 4,
    'palette': ['#e8a838',   # 1 shrubland  → amber
                '#78c679',   # 2 grassland  → green
                '#d9534f',   # 3 bare land  → red
                '#8c510a']   # 4 deforested → brown
}
Map.addLayer(degradation_type, type_vis, 'Degraded Land by Type (pre-exclusion)', False)

# ★ Final suitable degraded land
Map.addLayer(
    final_degraded,
    {'palette': ['#FF4500'], 'min': 1, 'max': 1},
    '★ Suitable Degraded Land (Solar PV)', True
)

# Legend
legend_dict = {
    'Suitable Degraded Land': '#FF4500',
    '  ↳ Shrubland':         '#e8a838',
    '  ↳ Grassland':         '#78c679',
    '  ↳ Bare land':         '#d9534f',
    '  ↳ Deforested (no recovery)': '#8c510a',
    'Exclusion zones':       '#808080',
}
Map.add_legend(title='Land Classification', legend_dict=legend_dict)

Map

Map(center=[-2.5, 118], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

In [20]:
# ── 8b. Export Interactive Map to HTML (pure Folium) ─────────────────────────
# geemap.foliumap has a compatibility bug with current xyzservices.
# This cell uses pure folium + GEE tile URLs instead — more reliable.
import os
import folium
from folium import plugins

# ── Helper: add a GEE image as a Folium tile layer ──────────────────────────
def add_ee_layer(fmap, ee_image, vis_params, name, show=True):
    map_id = ee.Image(ee_image).getMapId(vis_params)
    tile_url = map_id['tile_fetcher'].url_format
    folium.TileLayer(
        tiles=tile_url,
        attr='Google Earth Engine',
        name=name,
        overlay=True,
        control=True,
        show=show,
        opacity=0.8,
    ).add_to(fmap)

# ── Base map ─────────────────────────────────────────────────────────────────
fmap = folium.Map(
    location=[-2.5, 118],
    zoom_start=5,
    tiles='https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}',
    attr='Google Satellite Hybrid',
    name='Google Satellite',
    control=True,
)

# ── GEE layers ───────────────────────────────────────────────────────────────
add_ee_layer(
    fmap, worldcover,
    {'bands': ['Map'], 'min': 10, 'max': 100,
     'palette': ['006400','ffbb22','ffff4c','f096ff',
                 'fa0000','b4b4b4','f0f0f0','0064c8',
                 '0096a0','00cf75','fae6a0']},
    'ESA WorldCover 2021', show=False
)

add_ee_layer(
    fmap, hansen.select('lossyear').selfMask(),
    {'min': 1, 'max': 23, 'palette': ['yellow', 'red']},
    'Hansen Forest Loss 2001–2023', show=False
)

add_ee_layer(
    fmap, exclusion.selfMask(),
    {'palette': ['#808080'], 'min': 1, 'max': 1},
    'Exclusion Zones', show=False
)

add_ee_layer(
    fmap, degradation_type,
    {'min': 1, 'max': 4,
     'palette': ['#e8a838', '#78c679', '#d9534f', '#8c510a']},
    'Degraded Land by Type', show=False
)

add_ee_layer(
    fmap, final_degraded,
    {'palette': ['#FF4500'], 'min': 1, 'max': 1},
    '★ Suitable Degraded Land (Solar PV)', show=True
)

# ── Layer control ─────────────────────────────────────────────────────────────
folium.LayerControl(collapsed=False).add_to(fmap)

# ── Legend (injected as HTML) ─────────────────────────────────────────────────
legend_html = """
<div style="position: fixed; bottom: 30px; right: 30px; z-index: 9999;
            background: white; padding: 12px 16px; border-radius: 8px;
            border: 2px solid #ccc; font-size: 13px; line-height: 1.8;
            box-shadow: 2px 2px 6px rgba(0,0,0,0.3);">
  <b>Land Classification</b><br>
  <span style="background:#FF4500;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Suitable Degraded Land<br>
  <span style="background:#e8a838;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Shrubland<br>
  <span style="background:#78c679;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Grassland / Imperata<br>
  <span style="background:#d9534f;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Bare land<br>
  <span style="background:#8c510a;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Deforested (no recovery)<br>
  <span style="background:#808080;width:14px;height:14px;display:inline-block;margin-right:6px;border-radius:2px;"></span>Exclusion zones<br>
</div>
"""
fmap.get_root().html.add_child(folium.Element(legend_html))

# ── Save as standalone HTML ───────────────────────────────────────────────────
notebook_dir = os.path.dirname(os.path.abspath('GIS_degraded land.ipynb'))
html_path = os.path.join(notebook_dir, 'indonesia_degraded_land.html')
fmap.save(html_path)
print(f"HTML map saved → {html_path}")

fmap   # display inline

HTML map saved → /Users/rachman/Library/CloudStorage/GoogleDrive-rachman_setiawan@berkeley.edu/My Drive/01. ERG/5. Summer 2025/Rachman's Codes/Aa_Project/0_Thesis Project/0_Literally Random/WRI/01. Solar Boom - CI/indonesia_degraded_land.html


In [21]:
# ── 9. Area Statistics (national & by degradation type) ─────────────────────
pixel_area_km2 = ee.Image.pixelArea().divide(1e6)  # m² → km²

def get_area(mask, geometry=aoi, scale=500):
    result = (
        pixel_area_km2.updateMask(mask)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=geometry,
            scale=scale,
            maxPixels=1e13,
            bestEffort=True
        )
    )
    val = result.getInfo()
    return list(val.values())[0] if val else 0

print("Computing areas... (may take ~1 min)\n")

area_shrub  = get_area(shrubland.And(exclusion.Not()))
area_grass  = get_area(grassland.And(exclusion.Not()))
area_bare   = get_area(bare_land.And(exclusion.Not()))
area_defor  = get_area(deforested_no_recovery.And(exclusion.Not()))
area_total  = get_area(final_degraded)

rows = [
    ('Shrubland (class 20)',          area_shrub),
    ('Grassland/Imperata (class 30)', area_grass),
    ('Bare/sparse land (class 60)',   area_bare),
    ('Deforested, no recovery',       area_defor),
    ('TOTAL suitable degraded land',  area_total),
]
df_national = pd.DataFrame(rows, columns=['Category', 'Area (km²)'])
df_national['Area (km²)'] = df_national['Area (km²)'].round(0).astype(int)
df_national['Area (Mha)'] = (df_national['Area (km²)'] / 100).round(2)

print(df_national.to_string(index=False))

Computing areas... (may take ~1 min)



                     Category  Area (km²)  Area (Mha)
         Shrubland (class 20)        8659       86.59
Grassland/Imperata (class 30)       62904      629.04
  Bare/sparse land (class 60)        2378       23.78
      Deforested, no recovery       42254      422.54
 TOTAL suitable degraded land       55253      552.53


In [22]:
# ── 10. Area by Major Island ─────────────────────────────────────────────────
# Approximate bounding boxes for each major island group
islands = {
    'Sumatra':    ee.Geometry.Rectangle([ 95.0, -6.0, 109.0,  6.0]),
    'Java':       ee.Geometry.Rectangle([105.0, -9.0, 115.0, -5.0]),
    'Kalimantan': ee.Geometry.Rectangle([108.0, -5.0, 119.0,  7.5]),
    'Sulawesi':   ee.Geometry.Rectangle([119.0, -6.0, 126.0,  4.0]),
    'Maluku':     ee.Geometry.Rectangle([124.0, -9.0, 135.0,  4.0]),
    'NTB & NTT':  ee.Geometry.Rectangle([115.0,-11.0, 125.0, -7.0]),
    'Papua':      ee.Geometry.Rectangle([130.0, -9.0, 142.0,  1.0]),
}

print("Computing area by island... (may take ~2 min)\n")

island_rows = []
for name, bbox in islands.items():
    island_aoi = aoi.intersection(bbox, maxError=1000)
    km2 = get_area(final_degraded, geometry=island_aoi, scale=500)
    island_rows.append({'Island': name, 'Area (km²)': round(km2), 'Area (Mha)': round(km2/100, 2)})

df_islands = pd.DataFrame(island_rows)
df_islands.loc[len(df_islands)] = {
    'Island': 'TOTAL',
    'Area (km²)': df_islands['Area (km²)'].sum(),
    'Area (Mha)': round(df_islands['Area (km²)'].sum() / 100, 2)
}
print(df_islands.to_string(index=False))

Computing area by island... (may take ~2 min)

    Island  Area (km²)  Area (Mha)
   Sumatra       19803      198.03
      Java        1061       10.61
Kalimantan       23444      234.44
  Sulawesi        4316       43.16
    Maluku        2033       20.33
 NTB & NTT        3083       30.83
     Papua        3127       31.27
     TOTAL       56867      568.67


In [25]:
# ── 2. Solar Panel Potential Calculation ──────────────────────────────────
import pandas as pd
import numpy as np

# Data from your GIS analysis
island_data = {
    'Island': ['Sumatra', 'Java', 'Kalimantan', 'Sulawesi', 'Maluku', 'NTB & NTT', 'Papua'],
    'Area (km²)': [19803, 1061, 23444, 4316, 2033, 3083, 3127],
    'Area (Mha)': [198.03, 10.61, 234.44, 43.16, 20.33, 30.83, 31.27]
}

df = pd.DataFrame(island_data)

# ── Solar Panel Specifications ────────────────────────────────────────────
# Standard 650 Wp panel specifications
panel_power = 650  # Watts peak
panel_length = 2.108  # meters (typical monocrystalline)
panel_width = 1.048  # meters
panel_area = panel_length * panel_width  # m²
panel_area_hectares = panel_area / 10000  # convert to hectares

print("=" * 80)
print("SOLAR PANEL SPECIFICATIONS (650 Wp)")
print("=" * 80)
print(f"Power output: {panel_power} Wp")
print(f"Panel dimensions: {panel_length}m × {panel_width}m")
print(f"Panel area: {panel_area:.4f} m²")
print(f"Panel area: {panel_area_hectares:.6f} hectares")
print()

# ── Installation Density & Efficiency ────────────────────────────────────
# Assumptions for ground-mounted solar farms:
# - 30% of area is usable (accounting for roads, spacing, inverters, etc.)
usable_percentage = 0.30

print("=" * 80)
print("INSTALLATION ASSUMPTIONS")
print("=" * 80)
print(f"Usable land percentage: {usable_percentage*100}%")
print("(accounts for: inverters, equipment, roads, shade clearance spacing)")
print()

# ── Calculate Solar Panel Potential ──────────────────────────────────────
df['Usable Area (Mha)'] = df['Area (Mha)'] * usable_percentage
df['Usable Area (ha)'] = df['Usable Area (Mha)'] * 1_000_000
df['Panels per Island'] = (df['Usable Area (ha)'] / panel_area_hectares).astype(int)
df['Total Power (MW)'] = (df['Panels per Island'] * panel_power) / 1_000_000
df['Total Power (GW)'] = df['Total Power (MW)'] / 1000

# ── Display Results ──────────────────────────────────────────────────────
print("=" * 80)
print("SOLAR PANEL INSTALLATION POTENTIAL BY ISLAND")
print("=" * 80)
print()

results_df = df[['Island', 'Area (Mha)', 'Usable Area (Mha)', 'Panels per Island', 'Total Power (GW)']].copy()
results_df['Area (Mha)'] = results_df['Area (Mha)'].apply(lambda x: f"{x:,.2f}")
results_df['Usable Area (Mha)'] = results_df['Usable Area (Mha)'].apply(lambda x: f"{x:,.2f}")
results_df['Panels per Island'] = results_df['Panels per Island'].apply(lambda x: f"{x:,}")
results_df['Total Power (GW)'] = results_df['Total Power (GW)'].apply(lambda x: f"{x:,.2f}")

print(results_df.to_string(index=False))
print()

# ── National Totals ──────────────────────────────────────────────────────
total_degraded_area = df['Area (Mha)'].sum()
total_usable_area = df['Usable Area (Mha)'].sum()
total_panels = df['Panels per Island'].sum()
total_power_mw = df['Total Power (MW)'].sum()
total_power_gw = df['Total Power (GW)'].sum()

print("=" * 80)
print("NATIONAL TOTALS")
print("=" * 80)
print(f"Total degraded land: {total_degraded_area:,.2f} Mha")
print(f"Total usable area (30%): {total_usable_area:,.2f} Mha")
print(f"Total solar panels (650 Wp each): {total_panels:,}")
print(f"Total installed capacity: {total_power_mw:,.0f} MW = {total_power_gw:,.2f} GW")
print()

# ── Energy Production Estimation ────────────────────────────────────────
print("=" * 80)
print("ANNUAL ENERGY PRODUCTION ESTIMATE")
print("=" * 80)

psh = 4.5  # Peak Sun Hours per day (Indonesia average)
capacity_factor = psh / 24

annual_energy_gwh = total_power_gw * psh * 365 / 1000

print(f"Peak Sun Hours (PSH): {psh} hours/day (Indonesia average)")
print(f"Capacity factor: {capacity_factor*100:.1f}%")
print(f"Annual energy production: {annual_energy_gwh:,.0f} GWh")
print(f"Annual energy production: {annual_energy_gwh/1000:,.2f} TWh")
print()

# ── Context & Comparison ────────────────────────────────────────────────
print("=" * 80)
print("CONTEXT & COMPARISON")
print("=" * 80)
print()

indo_consumption_twh = 300
percentage_of_consumption = (annual_energy_gwh / 1000) / indo_consumption_twh * 100

print(f"Indonesia total electricity consumption: ~{indo_consumption_twh} TWh/year")
print(f"Solar potential represents: {percentage_of_consumption:.1f}% of national consumption")
print()

co2_per_gwh = 600
co2_avoided = annual_energy_gwh * co2_per_gwh

print(f"CO2 emissions avoided: {co2_avoided:,.0f} tonnes/year")
print(f"CO2 emissions avoided: {co2_avoided/1e6:,.2f} million tonnes/year")
print()

capex_per_wp = 0.80
total_capex_usd = total_panels * panel_power * capex_per_wp / 1e9

print(f"Estimated capital cost (@${capex_per_wp}/Wp): ${total_capex_usd:,.2f} billion USD")
print()

print("=" * 80)
print("NOTES:")
print("=" * 80)
print("• 30% usable area accounts for equipment spacing, roads, shade clearance")
print("• 650 Wp panel size: 2.108m × 1.048m (typical monocrystalline)")
print(f"• Indonesian solar irradiance: {psh} PSH/day (4.5-5 kWh/m²/day)")
print("• CO2 avoidance based on coal plant displacement")
print("• Capital cost is rough estimate; varies by location & installer")
print("=" * 80)

SOLAR PANEL SPECIFICATIONS (650 Wp)
Power output: 650 Wp
Panel dimensions: 2.108m × 1.048m
Panel area: 2.2092 m²
Panel area: 0.000221 hectares

INSTALLATION ASSUMPTIONS
Usable land percentage: 30.0%
(accounts for: inverters, equipment, roads, shade clearance spacing)

SOLAR PANEL INSTALLATION POTENTIAL BY ISLAND

    Island Area (Mha) Usable Area (Mha) Panels per Island Total Power (GW)
   Sumatra     198.03             59.41   268,918,297,434       174,796.89
      Java      10.61              3.18    14,408,034,821         9,365.22
Kalimantan     234.44             70.33   318,361,892,898       206,935.23
  Sulawesi      43.16             12.95    58,609,875,863        38,096.42
    Maluku      20.33              6.10    27,607,478,598        17,944.86
 NTB & NTT      30.83              9.25    41,866,136,999        27,212.99
     Papua      31.27              9.38    42,463,642,684        27,601.37

NATIONAL TOTALS
Total degraded land: 568.67 Mha
Total usable area (30%): 170.60 Mha


In [24]:
# ── 11. Export to Google Drive ───────────────────────────────────────────────
# Exports the binary suitable-degraded-land raster at 100 m resolution.
# Check progress at: https://code.earthengine.google.com/tasks

export_image = (
    final_degraded                          # 1 = suitable degraded land
    .unmask(0)                              # 0 = not degraded / excluded
    .toUint8()
)

task = ee.batch.Export.image.toDrive(
    image        = export_image,
    description  = 'Indonesia_Degraded_Land_SolarPV_Step1',
    folder       = 'GEE_Exports',
    fileNamePrefix = 'indonesia_degraded_land_100m',
    region       = aoi,
    scale        = 100,          # 100 m pixel size
    crs          = 'EPSG:4326',
    maxPixels    = 1e13,
    fileFormat   = 'GeoTIFF'
)
task.start()
print(f"Export task started. Task ID: {task.id}")
print("Monitor at: https://code.earthengine.google.com/tasks")

Export task started. Task ID: ULTMPYDTDYKXG46DI6LO4ZSC
Monitor at: https://code.earthengine.google.com/tasks


---
## ✅ Step 1 Complete — What You Have Now

| Output | Description |
|---|---|
| `final_degraded` | GEE image: 1 = degraded land suitable for Solar PV |
| `degradation_type` | GEE image: type-coded (1=shrub, 2=grass, 3=bare, 4=deforested) |
| `df_national` | DataFrame: area by degradation category (km² / Mha) |
| `df_islands` | DataFrame: area by major island |
| GeoTIFF (Drive) | 100 m raster ready for QGIS / further analysis |

## ➡️ Step 2 Preview — Solar PV Potential

Once you confirm the degraded land map looks correct, Step 2 will:
1. Load **GHI (Global Horizontal Irradiance)** from ERA5 or PVGIS
2. Overlay with `final_degraded` to get irradiance on degraded pixels only
3. Calculate **technical potential** (MW / GWh) using:
   - Panel efficiency (e.g. 20%)
   - Ground Coverage Ratio (e.g. 0.4)
   - Performance Ratio (e.g. 0.8)
4. Produce province-level potential maps and a final summary table